#  AI Personal Tutor — Server


## 1. Install dependencies

In [ ]:
!pip install -q faiss-cpu langchain langchain-community langchain-core langchain-classic langchain-huggingface sentence-transformers transformers==4.52.4 accelerate fastapi uvicorn pyngrok python-multipart pymupdf bitsandbytes docx2txt ebooklib beautifulsoup4

## 2. Log in to Hugging Face

In [ ]:
from huggingface_hub import login
login()

## 3. Load the model
Qwen2.5-7B-Instruct (strong in both English and Arabic), loaded in 4-bit NF4 so it fits in GPU memory (~5.5 GB instead of ~15 GB). `generate_text` applies the model's chat template and returns only the newly generated text.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import re
import torch

model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")

SYSTEM_PROMPT = ("You are a helpful tutor. Reply only in the language the user asks for. "
                 "Never write in Chinese or in any language that was not requested.")

def generate_text(prompt, max_new_tokens=1000, tries=2):
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}]
    chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat, return_tensors="pt").to(model.device)
    for attempt in range(tries):
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=True, temperature=0.3, top_p=0.9,
                                 repetition_penalty=1.05,
                                 pad_token_id=tokenizer.eos_token_id)
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        if not re.search(r"[\u4e00-\u9fff]", text):
            return text
        print("Found Chinese text in the reply, generating again...")
    # Last resort: remove any Chinese characters that are left
    return re.sub(r"[\u3000-\u303f\u4e00-\u9fff\uff00-\uffef]+", "", text).strip()

## 4. LangChain wrapper
Wraps `generate_text` as a LangChain LLM. `llm` handles long outputs (explanations, quizzes) and `llm_short` handles short ones (grading, topic suggestions).

In [ ]:
from langchain_core.language_models.llms import LLM
from typing import Any

class CustomHFLLM(LLM):
    max_new_tokens: int = 1000

    def _call(self, prompt: str, stop: Any = None) -> str:
        return generate_text(prompt, self.max_new_tokens)

    @property
    def _llm_type(self) -> str:
        return "custom_huggingface"

llm = CustomHFLLM()
llm_short = CustomHFLLM(max_new_tokens=300)

## 5. RAG: loading and indexing the material
- Loads PDF, TXT, DOCX and EPUB files.
- Removes lines repeated on most pages (headers, footers).
- Splits the text into chunks and indexes them in FAISS using multilingual embeddings.
- Detects the material's language and flags Arabic PDFs, whose extracted text is often noisy.

In [ ]:
import os
import re
from collections import Counter
from langchain_core.documents import Document
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Filled in when the student uploads material
vectordb = None
all_chunks = []
book_name = None
book_language = "English"
book_warning = None
suggested_topics = []

def load_epub(path):
    from ebooklib import epub, ITEM_DOCUMENT
    from bs4 import BeautifulSoup
    book = epub.read_epub(path)
    docs = []
    for item in book.get_items_of_type(ITEM_DOCUMENT):
        text = BeautifulSoup(item.get_content(), "html.parser").get_text("\n")
        if text.strip():
            docs.append(Document(page_content=text, metadata={"source": path}))
    return docs

def load_documents(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        return PyMuPDFLoader(path).load()
    if ext == ".txt":
        return TextLoader(path, encoding="utf-8").load()
    if ext == ".docx":
        return Docx2txtLoader(path).load()
    if ext == ".epub":
        return load_epub(path)
    raise ValueError("Supported files: PDF, TXT, DOCX and EPUB.")

def remove_repeated_lines(documents):
    if len(documents) <= 5:
        return documents
    counter = Counter()
    for doc in documents:
        lines = set(line.strip() for line in doc.page_content.split("\n") if line.strip())
        counter.update(lines)
    repeated = {line for line, count in counter.items() if count > len(documents) * 0.5}
    for doc in documents:
        kept = [line for line in doc.page_content.split("\n") if line.strip() not in repeated]
        doc.page_content = "\n".join(kept)
    return documents

def detect_language(text):
    arabic = len(re.findall(r"[\u0600-\u06FF]", text))
    letters = len(re.findall(r"[A-Za-z\u0600-\u06FF]", text))
    return "Arabic" if letters and arabic / letters > 0.3 else "English"

ARABIC_PDF_WARNING = (
    "Arabic PDFs often come out with merged words and mixed-up order, because a PDF stores "
    "letters as positions on the page, not as words. For better results, upload the same "
    "material as DOCX, EPUB or TXT (from the original file, not copied out of a PDF)."
)

def build_book(path):
    global vectordb, all_chunks, book_name, book_language, book_warning
    documents = remove_repeated_lines(load_documents(path))
    chunks = splitter.split_documents(documents)
    if not chunks:
        raise ValueError("Couldn't find any text in this file. If it's a scanned PDF, it needs OCR first.")
    vectordb = FAISS.from_documents(chunks, embedding)
    all_chunks = chunks
    book_name = os.path.basename(path)
    book_language = detect_language(" ".join(c.page_content for c in chunks[:20]))
    is_pdf = path.lower().endswith(".pdf")
    book_warning = ARABIC_PDF_WARNING if (is_pdf and book_language == "Arabic") else None
    return len(chunks)

def get_docs(topic, k=3):
    if vectordb is None:
        raise ValueError("No material uploaded yet.")
    return vectordb.similarity_search(topic, k=k)

def get_context(topic, k=3):
    return "\n\n".join(doc.page_content for doc in get_docs(topic, k))

## 6. Language hints
A final instruction added to the end of every prompt, written in the material's language. Small models follow the last instruction they read most closely.

In [ ]:
HINTS = {
    "explain": {
        "Arabic": "مهم جدًا: اكتب الشرح باللغة العربية فقط، مع إبقاء عناوين الأقسام (Explanation, Example, Tip for you) بالإنجليزية كما هي.",
        "English": "Important: write everything in English.",
    },
    "quiz": {
        "Arabic": "مهم جدًا: اكتب كل الأسئلة والإجابات باللغة العربية فقط، واسأل عن الأفكار مباشرة بدون ذكر كلمة \"النص\" أو \"السياق\".",
        "English": "Important: write everything in English, and ask about the ideas directly without saying \"the text\" or \"the context\".",
    },
    "eval": {
        "Arabic": "مهم جدًا: اكتب الـ feedback باللغة العربية فقط، مع إبقاء مفاتيح JSON بالإنجليزية.",
        "English": "Important: write the feedback in English.",
    },
    "topics": {
        "Arabic": "مهم جدًا: اكتب أسماء المواضيع باللغة العربية فقط، مع إبقاء مفاتيح JSON بالإنجليزية.",
        "English": "Important: write the topic names in English.",
    },
}

def hint(kind, language):
    return HINTS[kind][language]

## 7. Student memory
Stores scores and feedback per topic. Used to find weak topics and to personalize explanations.

In [ ]:
memory = {}

def save_result(topic, score, feedback):
    topic = topic.strip().lower()
    if topic not in memory:
        memory[topic] = {"scores": [], "feedbacks": []}
    memory[topic]["scores"].append(score)
    memory[topic]["feedbacks"].append(feedback)

def topic_average(topic):
    scores = memory[topic]["scores"]
    return sum(scores) / len(scores)

def weak_topics(limit=6):
    return [t for t in memory if topic_average(t) < limit]

def get_student_notes(topic):
    topic = topic.strip().lower()
    if topic not in memory:
        return "This is the student's first time studying this topic."
    avg = topic_average(topic)
    last_feedbacks = " | ".join(memory[topic]["feedbacks"][-3:])
    if avg < 6:
        return (f"The student struggled with this topic before (average {avg:.1f}/10). "
                f"Focus on fixing these mistakes: {last_feedbacks}")
    return (f"The student did well on this topic before (average {avg:.1f}/10). "
            "Give a slightly more advanced explanation.")

## 8. Explanation chain
Explains a topic from the retrieved context, adapted to the student's past results.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

explain_prompt = PromptTemplate(
    input_variables=["topic", "context", "student_notes", "language", "language_hint"],
    template="""You are a friendly tutor teaching a beginner student.

Context from the study material:
{context}

Topic: {topic}
Notes about this student: {student_notes}

Instructions:
- Explain the topic using ONLY the context above.
- If the context does not cover the topic, say so clearly.
- The student cannot see figures, tables or page numbers. Never mention them.
- Write your answer in {language}.
- Adapt your explanation to the notes about the student.
- The context may contain exercise questions. IGNORE them completely.
- Write ONLY three sections, in this format:
Explanation: (a simple explanation in a few short paragraphs)
Example: (one short, clear example. If the topic is about programming, use a short code example.)
Tip for you: (one or two sentences based on the notes about the student. If they made mistakes before, mention them directly.)
- Stop right after the Tip. Do not write anything else.
{language_hint}"""
)

explain_chain = LLMChain(llm=llm, prompt=explain_prompt)

def explain_topic(topic):
    context = get_context(topic)
    language = detect_language(context)
    return explain_chain.run({"topic": topic, "context": context,
                              "student_notes": get_student_notes(topic),
                              "language": language,
                              "language_hint": hint("explain", language)})

## 9. Quiz chain
Generates up to 10 short-answer questions at a chosen difficulty. Questions are created in batches of 5, each from different chunks, so they cover more of the material without repeating. A structured output parser turns the reply into lists of questions and answers.

In [ ]:
import math
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

questions_schema = ResponseSchema(name="questions", type="List[string]",
    description="A list of short-answer questions about the topic.")
answers_schema = ResponseSchema(name="answers", type="List[string]",
    description="A list of correct answers, in the same order as the questions.")

quiz_parser = StructuredOutputParser.from_response_schemas([questions_schema, answers_schema])
quiz_format = quiz_parser.get_format_instructions()

DIFFICULTY = {
    "easy": "simple recall questions about definitions and basic facts.",
    "medium": "questions that check understanding, like explaining how or why something works.",
    "hard": "questions that require applying the concept to a new situation or example.",
}
MAX_QUESTIONS = 10
BATCH_SIZE = 5
CHUNKS_PER_BATCH = 3

quiz_prompt = PromptTemplate(
    input_variables=["topic", "context", "num_questions", "difficulty", "difficulty_desc",
                     "language", "avoid", "format_instructions", "language_hint"],
    template="""You are a tutor creating a quiz for a student.

Context from the study material:
{context}

Instructions:
- Create exactly {num_questions} NEW short-answer questions about the topic: {topic}
- Difficulty: {difficulty} ({difficulty_desc})
- Use ONLY the context above. Do not copy the exercise questions from it.
- Do not repeat any of these earlier questions: {avoid}
- The student cannot see figures or tables. Every question must make sense on its own, without mentioning figures, tables or page numbers.
- Ask about the ideas directly. Never mention "the context", "the text" or "the passage".
- Write the questions and answers in {language}, but keep the JSON keys in English.
- Each answer should be one or two sentences.
- Respond ONLY with JSON, no extra text.
{format_instructions}
{language_hint}"""
)

quiz_chain = LLMChain(llm=llm, prompt=quiz_prompt)

def make_quiz_batch(topic, context, num_questions, difficulty, language, avoid, tries=3):
    for attempt in range(tries):
        text = quiz_chain.run({"topic": topic, "context": context,
                               "num_questions": num_questions,
                               "difficulty": difficulty,
                               "difficulty_desc": DIFFICULTY[difficulty],
                               "language": language,
                               "avoid": avoid,
                               "format_instructions": quiz_format,
                               "language_hint": hint("quiz", language)})
        try:
            quiz = quiz_parser.parse(text)
            n = min(len(quiz["questions"]), len(quiz["answers"]), num_questions)
            if n > 0:
                return {"questions": quiz["questions"][:n], "answers": quiz["answers"][:n]}
        except Exception:
            pass
        print("Parsing failed, trying again...")
    return None

def make_quiz(topic, num_questions=3, difficulty="medium"):
    num_questions = max(1, min(num_questions, MAX_QUESTIONS))
    n_batches = math.ceil(num_questions / BATCH_SIZE)
    docs = get_docs(topic, k=CHUNKS_PER_BATCH * n_batches)
    language = detect_language(" ".join(d.page_content for d in docs))

    questions, answers = [], []
    for b in range(n_batches):
        need = min(BATCH_SIZE, num_questions - len(questions))
        if need <= 0:
            break
        group = docs[b * CHUNKS_PER_BATCH:(b + 1) * CHUNKS_PER_BATCH] or docs[:CHUNKS_PER_BATCH]
        context = "\n\n".join(d.page_content for d in group)
        avoid = "; ".join(questions) if questions else "none"
        batch = make_quiz_batch(topic, context, need, difficulty, language, avoid)
        if batch:
            questions += batch["questions"]
            answers += batch["answers"]

    if not questions:
        return None
    return {"questions": questions[:num_questions], "answers": answers[:num_questions]}

## 10. Grading chain
Scores an answer from 0 to 10 and gives feedback. Empty answers get 0 without calling the model.

In [ ]:
score_schema = ResponseSchema(name="score",
    description="An integer score from 0 to 10 for the student's answer.", type="integer")
feedback_schema = ResponseSchema(name="feedback",
    description="Short, friendly feedback: what was correct and what was missing or wrong.")

eval_parser = StructuredOutputParser.from_response_schemas([score_schema, feedback_schema])
eval_format = eval_parser.get_format_instructions()

EMPTY_FEEDBACK = {
    "Arabic": "لم تكتب إجابة لهذا السؤال. حاول أن تجيب في المرة القادمة حتى لو لم تكن متأكدًا.",
    "English": "You didn't write an answer. Give it a try next time, even if you're not sure!",
}

eval_prompt = PromptTemplate(
    input_variables=["question", "correct_answer", "student_answer", "language",
                     "format_instructions", "language_hint"],
    template="""You are a fair tutor grading a student's answer.

Question: {question}
Correct answer: {correct_answer}
Student answer: {student_answer}

Instructions:
- Compare the student answer with the correct answer.
- Give a score from 0 to 10. Focus on the meaning, not the exact words.
- If the answer is unrelated to the question, a joke, nonsense, or says the student doesn't know, the score is 0.
- Only say an answer is "close" if it really contains part of the correct idea.
- Write short, friendly feedback speaking directly to the student (use "you").
- Write the feedback in {language}, but keep the JSON keys in English.
- Respond ONLY with JSON, no extra text.
{format_instructions}
{language_hint}"""
)

eval_chain = LLMChain(llm=llm_short, prompt=eval_prompt)

def evaluate_answer(question, correct_answer, student_answer, tries=3):
    language = detect_language(question + " " + correct_answer)
    if not student_answer.strip():
        return {"score": 0, "feedback": EMPTY_FEEDBACK[language]}
    for attempt in range(tries):
        text = eval_chain.run({"question": question,
                               "correct_answer": correct_answer,
                               "student_answer": student_answer,
                               "language": language,
                               "format_instructions": eval_format,
                               "language_hint": hint("eval", language)})
        try:
            result = eval_parser.parse(text)
            result["score"] = max(0, min(10, int(result["score"])))
            return result
        except Exception:
            print("Parsing failed, trying again...")
    return None

## 11. Topic suggestions
Samples chunks from across the whole material and asks the model for 6 topics to study.

In [ ]:
topics_schema = ResponseSchema(name="topics", type="List[string]",
    description="A list of 6 short topic names, 2 to 4 words each.")
topics_parser = StructuredOutputParser.from_response_schemas([topics_schema])
topics_format = topics_parser.get_format_instructions()

topics_prompt = PromptTemplate(
    input_variables=["sample", "language", "format_instructions", "language_hint"],
    template="""You are a tutor looking at parts of a student's study material.

Parts of the material:
{sample}

Instructions:
- List 6 main topics a student could study from this material.
- Each topic is a short name of 2 to 4 words, like a section title.
- Only list subjects a student would study. Never list details about the book itself (title, author, translation, publisher, table of contents).
- Write the topics in {language}, but keep the JSON keys in English.
- Respond ONLY with JSON, no extra text.
{format_instructions}
{language_hint}"""
)

topics_chain = LLMChain(llm=llm_short, prompt=topics_prompt)

def suggest_topics(tries=2):
    if not all_chunks:
        return []
    skip = len(all_chunks) // 20   # skip the first and last 5% (cover, copyright page, index)
    body = all_chunks[skip:len(all_chunks) - skip] or all_chunks
    step = max(1, len(body) // 8)
    sample = "\n---\n".join(c.page_content[:400] for c in body[::step][:8])
    for attempt in range(tries):
        text = topics_chain.run({"sample": sample, "language": book_language,
                                 "format_instructions": topics_format,
                                 "language_hint": hint("topics", book_language)})
        try:
            topics = topics_parser.parse(text)["topics"]
            if isinstance(topics, str):
                topics = [topics]
        except Exception:
            # Backup plan: take every text between quotes, except the key name itself
            topics = [t for t in re.findall(r'"([^"]+)"', text) if t != "topics"]
        topics = [str(t).strip() for t in topics if str(t).strip()]
        if topics:
            return topics[:6]
        print("Couldn't read the topics, trying again...")
    return []

## 12. API
| Endpoint | Description |
|---|---|
| `POST /upload` | Upload material, build the index, reset progress, and suggest topics |
| `POST /explain` | Explain a topic |
| `POST /quiz` | Create a quiz with a chosen number of questions and difficulty |
| `POST /evaluate` | Grade one answer and save the result |
| `GET /progress` | Material name, warning, suggested topics and progress |

Every request must include the access key, which is generated randomly each time the server starts.

In [ ]:
import secrets
from fastapi import FastAPI, Request, HTTPException, UploadFile, File

API_KEY = secrets.token_urlsafe(8)
app = FastAPI()

def check_key(req):
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")

def check_book():
    if vectordb is None:
        raise HTTPException(status_code=400, detail="Upload a book or lecture notes first.")

@app.post("/upload")
async def upload(req: Request, file: UploadFile = File(...)):
    global memory, suggested_topics
    check_key(req)
    path = os.path.join("/kaggle/working", os.path.basename(file.filename))
    with open(path, "wb") as f:
        f.write(await file.read())
    try:
        n_chunks = build_book(path)
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    memory = {}   # new material = new progress
    suggested_topics = suggest_topics()
    return {"book": book_name, "chunks": n_chunks, "language": book_language,
            "warning": book_warning, "topics": suggested_topics}

@app.post("/explain")
async def explain(req: Request):
    check_key(req)
    check_book()
    data = await req.json()
    return {"topic": data["topic"], "explanation": explain_topic(data["topic"])}

@app.post("/quiz")
async def quiz(req: Request):
    check_key(req)
    check_book()
    data = await req.json()
    num = max(1, min(int(data.get("num_questions", 3)), MAX_QUESTIONS))
    difficulty = data.get("difficulty", "medium")
    if difficulty not in DIFFICULTY:
        difficulty = "medium"
    result = make_quiz(data["topic"], num, difficulty)
    if result is None:
        raise HTTPException(status_code=500, detail="Couldn't create the quiz. Try again.")
    return {"topic": data["topic"], "difficulty": difficulty, **result}

@app.post("/evaluate")
async def evaluate(req: Request):
    check_key(req)
    data = await req.json()
    result = evaluate_answer(data["question"], data["correct_answer"], data["student_answer"])
    if result is None:
        return {"score": None, "feedback": "Couldn't grade this answer."}
    save_result(data["topic"], result["score"], result["feedback"])
    return result

@app.get("/progress")
async def progress(req: Request):
    check_key(req)
    topics = {t: {"average": topic_average(t), "count": len(memory[t]["scores"])} for t in memory}
    return {"book": book_name, "warning": book_warning, "suggested": suggested_topics,
            "topics": topics, "weak_topics": weak_topics()}

## 13. Start the server
Asks for your ngrok token, then prints the **server link** and **access key**. Paste both into the app under **🔌 Server connection**.

In [ ]:
import uvicorn, threading, time, socket
from getpass import getpass
from pyngrok import ngrok, conf

NGROK_TOKEN = getpass("Paste your ngrok token: ")

def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url

def run():
    uvicorn.run(app, host="0.0.0.0", port=port)

threading.Thread(target=run, daemon=True).start()
time.sleep(1)
print("Server link:", public_url)
print("Access key: ", API_KEY)